# <span style="color:darkblue"> Project 1 and 2: Climate Adaptation Fund Projects </span>

<font size = 4>
By: Tanya Jagdish

<b> Import packages

In [1]:
#import scripts
exec(open("./scripts/import_packages.py").read())

# I. Initialize Web Driver

In [2]:
# Open browser to start web scraping
opts = Options()
service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=opts)

# Navigate to specific website
starting_url = 'https://www.adaptation-fund.org/projects-programmes/project-information/projects-table-view/'
driver.get(starting_url)

<b> Extract first row

In [3]:
# find the first row of the table
first_row = driver.find_element('xpath', '//*[@id="projects-table"]/tbody/tr[1]')

# extract the HTML and make it readable
html_code = first_row.get_attribute("outerHTML")
parse_code = BeautifulSoup(html_code, "html.parser").prettify()

# Print it to see the structure
print(parse_code)

<tr class="odd" role="row">
 <td>
  <a href="https://www.adaptation-fund.org/project/transforming-communities-a-nexus-of-climate-smart-agriculture-livelihood-diversification-and-womens-economic-empowerment/">
   Transforming Communities: A Nexus of Climate-Smart Agriculture, Livelihood Diversification, and Women’s Economic Empowerment
  </a>
 </td>
 <td>
  Ministry of Finance and Economic Cooperation of the Federal Democratic Republic of Ethiopia
 </td>
 <td>
  Ethiopia
 </td>
 <td class="dt-right">
  9,999,328
 </td>
 <td class="dt-right">
  0
 </td>
 <td>
  Multi-sector
 </td>
</tr>



In [4]:
# Find all cells (td tags) in the first row
cells = first_row.find_elements('xpath', './/td')

# Print how many cells we found
print(f"Number of cells: {len(cells)}")

# Extract and print text from each cell
for i, cell in enumerate(cells):
    print(f"Cell {i}: {cell.text}")

# This would work but you wouldn't know which cell is which
for cell in cells:
    print(cell.text)

Number of cells: 6
Cell 0: Transforming Communities: A Nexus of Climate-Smart Agriculture, Livelihood Diversification, and Women’s Economic Empowerment
Cell 1: Ministry of Finance and Economic Cooperation of the Federal Democratic Republic of Ethiopia
Cell 2: Ethiopia
Cell 3: 9,999,328
Cell 4: 0
Cell 5: Multi-sector
Transforming Communities: A Nexus of Climate-Smart Agriculture, Livelihood Diversification, and Women’s Economic Empowerment
Ministry of Finance and Economic Cooperation of the Federal Democratic Republic of Ethiopia
Ethiopia
9,999,328
0
Multi-sector


In [5]:
# Find all rows in the table body
all_rows = driver.find_elements('xpath', '//*[@id="projects-table"]/tbody/tr')

# Check how many rows we found
print(f"Number of rows found: {len(all_rows)}")



Number of rows found: 10


In [6]:
# Create an empty list to store all project data
data = []

# Loop through each row
for row in all_rows:
    # Find all cells in this row
    cells = row.find_elements('xpath', './/td')
    
    # Extract text from each cell and store as a dictionary
    project_data = {
        'project_name': cells[0].text,
        'implementing_entity': cells[1].text,
        'country': cells[2].text,
        'grant_amount': cells[3].text,
        'transferred_amount': cells[4].text,
        'sector': cells[5].text
    }
    
    # Add this project to our data list
    data.append(project_data)

# Check how many projects we collected
print(f"Collected {len(data)} projects")

# Look at the first project
print(data[0])

Collected 10 projects
{'project_name': 'Transforming Communities: A Nexus of Climate-Smart Agriculture, Livelihood Diversification, and Women’s Economic Empowerment', 'implementing_entity': 'Ministry of Finance and Economic Cooperation of the Federal Democratic Republic of Ethiopia', 'country': 'Ethiopia', 'grant_amount': '9,999,328', 'transferred_amount': '0', 'sector': 'Multi-sector'}


In [7]:
# Convert to DataFrame
adaptation_projects = pd.DataFrame(data)

# Check the shape (rows, columns)
print(f"\nDataFrame shape: {adaptation_projects.shape}")


DataFrame shape: (10, 6)


In [8]:
# Find the dropdown element (the select tag, not the label)
dropdown = driver.find_element('xpath', '//select[@name="projects-table_length"]')

# Convert to a Select object
from selenium.webdriver.support.ui import Select
select_dropdown = Select(dropdown)

# Select the option for 100 entries
select_dropdown.select_by_value('100')

# Wait for the page to reload
import time
time.sleep(3)

# Now count how many rows we have
all_rows_100 = driver.find_elements('xpath', '//*[@id="projects-table"]/tbody/tr')
print(f"Number of rows after selecting: {len(all_rows_100)}")

Number of rows after selecting: 100


In [9]:
# Create an empty list to store all project data
data_page1 = []

# Loop through each row (all 100 of them)
for row in all_rows_100:
    # Find all cells in this row
    cells = row.find_elements('xpath', './/td')
    
    # Extract text from each cell and store as a dictionary
    project_data = {
        'project_name': cells[0].text,
        'implementing_entity': cells[1].text,
        'country': cells[2].text,
        'grant_amount': cells[3].text,
        'transferred_amount': cells[4].text,
        'sector': cells[5].text
    }
    
    # Add this project to our data list
    data_page1.append(project_data)

# Check how many projects we collected
print(f"Collected {len(data_page1)} projects from page 1")

# Convert to DataFrame
df_page1 = pd.DataFrame(data_page1)
print(f"DataFrame shape: {df_page1.shape}")

Collected 100 projects from page 1
DataFrame shape: (100, 6)


<b> Extracting detailed information from project page </b>

Framework for approaching this problem:

For each project row in the table:

1. Find the link in that row

2. Click the link → Navigate to detail page

3. Extract ALL "At a Glance" information: 
   * Project Name (from the header) 
   * Country/Region 
   * Sector 
   * Grant Amount 
   * Implementing Entity
   * Executing Entity
   * Approval Date
   * Duration
   * Status
4. Navigate BACK to the main table
5. Move to next project row
6. Repeat


<b> Data Structure: </b>
Each project will be stored as a dictionary with ~9 fields (all from the detail page), then we'll convert the list of dictionaries to a DataFrame.

<b> Key technical challenges I had to figure out: </b>

* Navigating back to main table: will use driver.back() for this

* Need time.sleep() to let pages load

<b> Test code for one project 

In [17]:
# Get the first row
first_row = all_rows[0]

# Find the link in the first cell (the project name cell)
project_link = first_row.find_element('xpath', './/td[1]/a')

# Click the link
project_link.click()

# Wait for the detail page to load
time.sleep(3)

# Print the current URL to confirm we're on the detail page
print(f"Current URL: {driver.current_url}")

StaleElementReferenceException: Message: stale element reference: stale element not found
  (Session info: chrome=141.0.7390.108); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#stale-element-reference-exception
Stacktrace:
	GetHandleVerifier [0x0x38fe43+66515]
	GetHandleVerifier [0x0x38fe84+66580]
	(No symbol) [0x0x17dc48]
	(No symbol) [0x0x18fc3b]
	(No symbol) [0x0x18ed10]
	(No symbol) [0x0x1850e2]
	(No symbol) [0x0x183608]
	(No symbol) [0x0x1869d4]
	(No symbol) [0x0x186a68]
	(No symbol) [0x0x1c7f94]
	(No symbol) [0x0x1c8aab]
	(No symbol) [0x0x1bdc71]
	(No symbol) [0x0x1eb214]
	(No symbol) [0x0x1bdb74]
	(No symbol) [0x0x1eb384]
	(No symbol) [0x0x20cba7]
	(No symbol) [0x0x1eafc6]
	(No symbol) [0x0x1bc2ca]
	(No symbol) [0x0x1bd154]
	GetHandleVerifier [0x0x5e7353+2521315]
	GetHandleVerifier [0x0x5e22d3+2500707]
	GetHandleVerifier [0x0x3b7c94+229924]
	GetHandleVerifier [0x0x3a81f8+165768]
	GetHandleVerifier [0x0x3aec

In [18]:
# Find all the info boxes within "At a Glance"
info_boxes = driver.find_elements('xpath', '//div[@class="project-info-box"]')

print(f"Found {len(info_boxes)} info boxes")

# Create empty dictionary for this project
project_data = {}

# Loop through each info box
for box in info_boxes:
    try:
        # Extract label (h4)
        label = box.find_element('xpath', './/h4').text
        
        # Extract value from project-terms div (this gets all text inside)
        value = box.find_element('xpath', './/div[@class="project-terms"]').text
        
        # Clean the label
        clean_label = label.replace(":", "").replace("/", "_").replace(" ", "_").lower()
        
        # Store in dictionary
        project_data[clean_label] = value
        
    except Exception as e:
        print(f"Error extracting box: {e}")
        continue

# Print the extracted data
print(f"\nExtracted {len(project_data)} fields:")
for key, val in project_data.items():
    print(f"{key}: {val}")

Found 8 info boxes

Extracted 8 fields:
country_region: Ethiopia/Africa
sector: Multi-sector
grant_amount: USD 9,999,328
implementing_entity: Ministry of Finance and Economic Cooperation of the Federal Democratic Republic of Ethiopia
executing_entity: Ministry of Water and Energy,Ministry of Agriculture
approval_date: 10/10/2025
duration: 3 years
status: Proposal Approved
